# KinoCompass – EASE на GPU

EASE (Steck, 2019) – линейная модель «фильм × фильм» с решением в одну формулу. На всех ~21 тыс. фильмах-кандидатах её матрица весит около 2 ГБ, поэтому она обучается здесь, на GPU.

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`, `protocol.py`, `metrics.py`): те же люди, те же сценарии, те же метрики. Набор MovieLens 32M скачивается в память Colab.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `ease_<период>.zip` с отчётом (около 100 КБ) – его нужно положить в `model/reports/`.

Время на T4 (бесплатный Colab): загрузка набора и сценарии – 2 минуты, обучение пяти λ на GPU – 10 минут, проверка на GPU – 6 минут, всего 18 минут. Вкладку не закрывать до конца.

In [1]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [2]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [3]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

ce248cd (grafted, HEAD -> main, origin/main, origin/HEAD) Исправил повторный запуск ноутбука EASE в том же сеансе Colab


In [4]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py ease

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
сценарии: {'cold5': {'people': 3912, 'median_inputs': 5, 'median_targets': 44}, 'cold10': {'people': 3903, 'median_inputs': 10, 'median_targets': 41}, 'warm': {'people': 5677, 'median_inputs': 308, 'median_targets': 8}}
cold5 popular_year ndcg@10 0.34536
cold10 popular_year ndcg@10 0.30037
warm popular_year ndcg@10 0.08778
cold5 ease_l200 ndcg@10 0.33324
cold10 ease_l200 ndcg@10 0.30552
warm ease_l200 ndcg@10 0.10093
cold5 ease_l1000 ndcg@10 0.33508
cold10 ease_l1000 ndcg@10 0.30659
warm ease_l1000 ndcg@10 0.1051
cold5 ease_l5000 ndcg@10 0.33665
cold10 ease_l5000 ndcg@10 0.30684
warm ease_l5000 ndcg@10 0.10684
cold5 ease_l20000 ndcg@10 0.33747
cold10 ease_l20000 ndcg@10 0.30512
w

In [5]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/ease_{PERIOD}.zip reports/ease && ls -la reports/ease
files.download(f"/content/ease_{PERIOD}.zip")

total 176
drwxr-xr-x 2 root root   4096 Nov  4 21:00 .
drwxr-xr-x 6 root root   4096 Nov  4 21:00 ..
-rw-r--r-- 1 root root  12481 Nov  4 21:00 validation.json
-rw-r--r-- 1 root root 155090 Nov  4 21:00 validation_users.csv.gz
